# Lab 7: Production-Ready RAG Pipeline (Capstone)

**Level:** Capstone | **Duration:** ~60 minutes

## Objective
Build a production-grade RAG pipeline that combines everything from Labs 1-6:
- Multiple document sources
- Hybrid retrieval (semantic + keyword)
- Re-ranking
- Evaluation with RAGAS metrics
- Simple caching layer

## What's Different
This is a capstone lab. You get:
- The requirements and architecture
- Key scaffolding and helper functions
- Evaluation criteria

You build the implementation. Minimal hand-holding.

## Evaluation Criteria
Your pipeline will be evaluated on:
1. **Answer accuracy** — Does it answer correctly based on the sources?
2. **Faithfulness** — Does it avoid hallucination?
3. **Relevance** — Does it retrieve the right chunks?
4. **Latency** — Is it fast enough for interactive use (<5s)?
5. **Robustness** — Does it handle edge cases gracefully?

## Setup

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma chromadb sentence-transformers ragas rank_bm25

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-key-here"

import time
import hashlib
import numpy as np
from typing import Optional

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Setup complete!")

## Part 1: Multiple Document Sources

Your knowledge base has 3 different document sources. Each has different structure and metadata.

In [ ]:
# Source 1: Company Policies
policies = """
# SkyWing Airlines - Passenger Policies (2024)

## Refund Policy
Full refund available for cancellations made 24+ hours before departure on tickets purchased 7+ days prior. Light fares are non-refundable but convertible to 12-month travel credit. Flex and Premium fares are fully refundable. Cancellation fees: 75 EUR (short-haul) or 150 EUR (long-haul) for late cancellations. Refunds are processed within 7 business days to the original payment method.

## Baggage Policy
Cabin: one bag (55x40x20cm, 8kg) + one personal item for all passengers. Checked: Light = purchase required (from 15 EUR), Flex = one free bag (23kg), Premium = two free bags (23kg each). Excess: 15 EUR/kg first 9kg, 25 EUR/kg after. Sports equipment: 30-75 EUR. Musical instruments can be brought as cabin bag if within size limits, or a seat can be purchased.

## Delay Compensation
Under EU261: delays over 3 hours qualify for 250 EUR (under 1500km), 400 EUR (1500-3500km), or 600 EUR (over 3500km). Extraordinary circumstances (weather, strikes, security) are exempt. Claims must be filed within 2 years. SkyWing provides refreshments (2h+ delay), meal vouchers (4h+), and hotel accommodation (overnight delays).
"""

# Source 2: Technical Documentation
tech_docs = """
# SkyWing Airlines - Technical Systems Guide

## Booking System
SkyWing uses the Altea PSS (Passenger Service System) for reservations, inventory management, and departure control. The system processes approximately 50,000 bookings per day. API integrations are available through Web Services for corporate partners and travel agencies.

## Revenue Management
Dynamic pricing is managed through the Revenue Management system. The system uses 26 fare classes (A-Z) with real-time inventory controls. Pricing decisions are made based on: booking pace vs. forecast, days to departure, competitive fares, historical demand patterns. Overrides require revenue management analyst approval.

## Airport Operations
Departure control uses Altea DCS. Check-in, boarding, and load control are managed through a single platform. Weight and balance calculations are automated with manual override capability. Turnaround target: 35 minutes for narrow-body, 75 minutes for wide-body aircraft.

## Crew Management
Crew scheduling uses the Sky Suite system. Pairing optimization runs monthly, considering: regulatory rest requirements, qualification requirements, base constraints, crew preferences. Disruption management is automated for delays under 2 hours; longer disruptions require IOCC (Integrated Operations Control Center) intervention.
"""

# Source 3: FAQ
faq = """
# SkyWing Airlines - Frequently Asked Questions

Q: How do I check in online?
A: Online check-in opens 24 hours before departure. Visit skywing.com or use the SkyWing app. You'll need your booking reference and last name. Mobile boarding passes are accepted at all airports.

Q: Can I change my seat after booking?
A: Flex and Premium passengers can select or change seats for free. Light fare passengers can purchase a seat selection from 5 EUR. Seat changes can be made online up to 1 hour before departure.

Q: What happens if I miss my connecting flight due to a SkyWing delay?
A: SkyWing will automatically rebook you on the next available connection at no cost. If the delay causes an overnight stay, hotel and transport are provided. You may also be entitled to EU261 compensation.

Q: How do I add extra baggage after booking?
A: Extra baggage can be added online up to 2 hours before departure at a discounted rate (from 15 EUR). Airport pricing starts at 25 EUR. Premium members receive an additional free bag.

Q: Is Wi-Fi available on flights?
A: Wi-Fi is available on all long-haul flights and 60% of short-haul aircraft. Pricing: 5 EUR (1 hour), 10 EUR (3 hours), 15 EUR (full flight). Premium passengers get free Wi-Fi.

Q: How do I request special assistance?
A: Contact our special assistance team at least 48 hours before departure. Services include wheelchair assistance, priority boarding, and adapted seating. Service animals are allowed with prior notification.

Q: What is the SkyWing Miles loyalty program?
A: Earn 1-2 miles per km depending on fare class. Tier levels: Silver (25K miles), Gold (50K miles), Platinum (100K miles). Redeem miles for flights, upgrades, and partner services. Miles expire after 24 months of inactivity.

Q: Can I bring my pet on the flight?
A: Small cats and dogs (under 8kg with carrier) can travel in the cabin for 40 EUR per flight. Larger pets travel in the climate-controlled cargo hold for 75-150 EUR depending on route. Veterinary certificate required, issued within 10 days of travel.
"""

# Combine all sources with metadata
sources = [
    ("policies", policies),
    ("tech-docs", tech_docs),
    ("faq", faq),
]

print(f"Loaded {len(sources)} document sources.")
for name, text in sources:
    print(f"  {name}: {len(text)} chars, ~{len(text.split())} words")

## Part 2: Chunk and Index

Split all documents and index them. Preserve the source metadata.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

all_docs = []
all_texts = []  # For BM25

for source_name, text in sources:
    chunks = splitter.split_text(text)
    for i, chunk in enumerate(chunks):
        doc = Document(
            page_content=chunk,
            metadata={"source": source_name, "chunk_id": f"{source_name}_{i}"}
        )
        all_docs.append(doc)
        all_texts.append(chunk)

# Vector store for semantic search
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name="capstone_rag",
)

# BM25 index for keyword search
tokenized_corpus = [doc.lower().split() for doc in all_texts]
bm25 = BM25Okapi(tokenized_corpus)

print(f"Indexed {len(all_docs)} chunks total.")
for source_name, _ in sources:
    count = sum(1 for d in all_docs if d.metadata["source"] == source_name)
    print(f"  {source_name}: {count} chunks")

## Part 3: Hybrid Retrieval

**Your task:** Implement hybrid retrieval that combines semantic search (ChromaDB) with keyword search (BM25). The standard approach is Reciprocal Rank Fusion (RRF).

RRF formula: `score(doc) = sum(1 / (k + rank_in_list))` across all lists, where k=60 is standard.

In [ ]:
def semantic_search(query, k=20):
    """Semantic search using ChromaDB."""
    results = vectorstore.similarity_search_with_score(query, k=k)
    return [(doc, score) for doc, score in results]

def keyword_search(query, k=20):
    """BM25 keyword search."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:k]
    return [(all_docs[i], scores[i]) for i in top_indices if scores[i] > 0]

def hybrid_search(query, k=10, rrf_k=60):
    """Hybrid search using Reciprocal Rank Fusion."""
    # Get results from both methods
    semantic_results = semantic_search(query, k=20)
    keyword_results = keyword_search(query, k=20)

    # RRF scoring
    doc_scores = {}  # chunk_id -> rrf_score
    doc_map = {}     # chunk_id -> Document

    for rank, (doc, _) in enumerate(semantic_results):
        cid = doc.metadata["chunk_id"]
        doc_scores[cid] = doc_scores.get(cid, 0) + 1.0 / (rrf_k + rank + 1)
        doc_map[cid] = doc

    for rank, (doc, _) in enumerate(keyword_results):
        cid = doc.metadata["chunk_id"]
        doc_scores[cid] = doc_scores.get(cid, 0) + 1.0 / (rrf_k + rank + 1)
        doc_map[cid] = doc

    # Sort by RRF score
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)

    return [(doc_map[cid], score) for cid, score in sorted_docs[:k]]

# Test it
query = "What is the refund policy for Light fares?"
results = hybrid_search(query, k=5)

print(f"Hybrid search for: \"{query}\"\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. [RRF: {score:.4f}] [{doc.metadata['source']}] {doc.page_content[:80]}...")

## Part 4: Re-ranking

Apply cross-encoder re-ranking on the hybrid results.

In [ ]:
def rerank(query, documents, top_k=5):
    """Re-rank documents using cross-encoder."""
    if not documents:
        return []
    pairs = [[query, doc.page_content] for doc, _ in documents]
    scores = cross_encoder.predict(pairs)
    scored = list(zip(scores, [doc for doc, _ in documents]))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

def full_retrieve(query, initial_k=15, final_k=5):
    """Complete retrieval: hybrid search + re-ranking."""
    hybrid_results = hybrid_search(query, k=initial_k)
    reranked = rerank(query, hybrid_results, top_k=final_k)
    return [doc for _, doc in reranked]

# Test
docs = full_retrieve("What compensation am I entitled to for a 4-hour delay?")
print("Full retrieval (hybrid + re-rank):")
for i, doc in enumerate(docs, 1):
    print(f"  {i}. [{doc.metadata['source']}] {doc.page_content[:80]}...")

## Part 5: Simple Caching Layer

Implement a cache to avoid redundant LLM calls for repeated or near-identical questions.

In [ ]:
class SimpleRAGCache:
    """In-memory cache for RAG answers.
    Uses query hash for exact matches.
    """

    def __init__(self, max_size=100):
        self.cache = {}
        self.max_size = max_size
        self.hits = 0
        self.misses = 0

    def _hash(self, query: str) -> str:
        normalized = query.strip().lower()
        return hashlib.md5(normalized.encode()).hexdigest()

    def get(self, query: str) -> Optional[dict]:
        key = self._hash(query)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None

    def put(self, query: str, result: dict):
        if len(self.cache) >= self.max_size:
            # Evict oldest entry
            oldest_key = next(iter(self.cache))
            del self.cache[oldest_key]
        key = self._hash(query)
        self.cache[key] = result

    def stats(self):
        total = self.hits + self.misses
        rate = self.hits / total * 100 if total > 0 else 0
        return {"hits": self.hits, "misses": self.misses, "hit_rate": f"{rate:.1f}%"}

cache = SimpleRAGCache()
print("Cache initialized.")

## Part 6: The Complete Pipeline

Wire everything together into a single `ask()` function.

In [ ]:
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant for SkyWing Airlines. Answer the question based ONLY on the provided context.

Rules:
- Only use information from the context below
- If the context doesn't contain the answer, say "I don't have this information in our documentation."
- Cite which source(s) you used: [policies], [tech-docs], or [faq]
- Be concise but complete

Context:
{context}

Question: {question}

Answer:
""")

def ask(question: str, use_cache: bool = True) -> dict:
    """Full production RAG pipeline."""
    start_time = time.time()

    # Check cache
    if use_cache:
        cached = cache.get(question)
        if cached:
            cached["from_cache"] = True
            cached["latency"] = time.time() - start_time
            return cached

    # Retrieve (hybrid + re-rank)
    retrieved_docs = full_retrieve(question)

    # Generate
    context_parts = []
    sources_used = set()
    for doc in retrieved_docs:
        context_parts.append(f"[Source: {doc.metadata['source']}]\n{doc.page_content}")
        sources_used.add(doc.metadata["source"])

    context = "\n\n---\n\n".join(context_parts)

    answer = (answer_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question,
    })

    result = {
        "question": question,
        "answer": answer,
        "sources": list(sources_used),
        "num_chunks": len(retrieved_docs),
        "from_cache": False,
        "latency": time.time() - start_time,
        "context": context,  # Keep for evaluation
    }

    # Cache the result
    if use_cache:
        cache.put(question, result)

    return result

# Test
result = ask("What is the refund policy for Light fare tickets?")
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")
print(f"Latency: {result['latency']:.2f}s")
print(f"Cache: {result['from_cache']}")

## Part 7: Test the Cache

In [ ]:
# First call - cache miss
r1 = ask("How do I check in online?")
print(f"First call:  latency={r1['latency']:.3f}s, from_cache={r1['from_cache']}")

# Second call - cache hit (same question)
r2 = ask("How do I check in online?")
print(f"Second call: latency={r2['latency']:.3f}s, from_cache={r2['from_cache']}")

# Slightly different - cache miss (case sensitivity is handled)
r3 = ask("how do i check in online?")
print(f"Lowercase:   latency={r3['latency']:.3f}s, from_cache={r3['from_cache']}")

print(f"\nCache stats: {cache.stats()}")

## Part 8: Golden Test Set

Define a set of question-answer pairs with expected answers for evaluation.

In [ ]:
golden_test_set = [
    {
        "question": "What is the cancellation fee for a long-haul Light fare ticket?",
        "expected_answer": "Light fares are non-refundable but can be converted to 12-month travel credit.",
        "expected_source": "policies",
    },
    {
        "question": "How many checked bags does a Premium passenger get?",
        "expected_answer": "Premium passengers get two free checked bags, each up to 23kg.",
        "expected_source": "policies",
    },
    {
        "question": "What compensation do I get for a 4-hour flight delay?",
        "expected_answer": "For delays over 3 hours, EU261 compensation of 250-600 EUR depending on distance. For 4+ hour delays, a 15 EUR meal voucher is also provided.",
        "expected_source": "policies",
    },
    {
        "question": "What booking system does SkyWing use?",
        "expected_answer": "SkyWing uses the Altea PSS for reservations, inventory management, and departure control.",
        "expected_source": "tech-docs",
    },
    {
        "question": "Can I bring my cat on the plane?",
        "expected_answer": "Small cats under 8kg with carrier can travel in the cabin for 40 EUR per flight. Veterinary certificate required within 10 days of travel.",
        "expected_source": "faq",
    },
    {
        "question": "How do I change my seat on a Light fare?",
        "expected_answer": "Light fare passengers can purchase a seat selection from 5 EUR. Changes can be made online up to 1 hour before departure.",
        "expected_source": "faq",
    },
    {
        "question": "What is the turnaround time for narrow-body aircraft?",
        "expected_answer": "Turnaround target is 35 minutes for narrow-body aircraft.",
        "expected_source": "tech-docs",
    },
    {
        "question": "How do I earn miles in the loyalty program?",
        "expected_answer": "Earn 1 mile per km on Light fares, 1.5 on Flex, and 2 on Premium. Tier bonuses apply: Silver 25%, Gold 50%, Platinum 100%.",
        "expected_source": "faq",
    },
]

print(f"Golden test set: {len(golden_test_set)} question-answer pairs.")

## Part 9: Run the Pipeline on the Test Set

In [ ]:
results = []

for test_case in golden_test_set:
    result = ask(test_case["question"], use_cache=False)
    result["expected_answer"] = test_case["expected_answer"]
    result["expected_source"] = test_case["expected_source"]
    results.append(result)

    print(f"Q: {test_case['question']}")
    print(f"A: {result['answer'][:150]}...")
    print(f"Sources: {result['sources']} (expected: {test_case['expected_source']})")
    print(f"Latency: {result['latency']:.2f}s")
    print("-" * 60)

## Part 10: Evaluate with RAGAS

RAGAS (Retrieval Augmented Generation Assessment) provides standardized metrics for RAG evaluation.

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset

# Prepare data for RAGAS
eval_data = {
    "question": [r["question"] for r in results],
    "answer": [r["answer"] for r in results],
    "contexts": [[r["context"]] for r in results],
    "ground_truth": [r["expected_answer"] for r in results],
}

eval_dataset = Dataset.from_dict(eval_data)

print("Running RAGAS evaluation (this takes a minute)...")
ragas_result = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

print("\nRAGAS Evaluation Results")
print("=" * 40)
for metric, score in ragas_result.items():
    bar = "#" * int(score * 20) if isinstance(score, float) else ""
    print(f"  {metric:<25} {score:.4f}  {bar}")

## Part 11: Source Accuracy Check

Did the pipeline retrieve from the correct source for each question?

In [ ]:
correct_source = 0
total = len(results)

print(f"{'Question':<55} {'Expected':<12} {'Got':<20} {'Match'}")
print("-" * 100)

for r in results:
    expected = r["expected_source"]
    got = r["sources"]
    match = expected in got
    if match:
        correct_source += 1
    status = "OK" if match else "MISS"
    print(f"{r['question'][:53]:<55} {expected:<12} {str(got):<20} {status}")

print(f"\nSource accuracy: {correct_source}/{total} ({correct_source/total*100:.0f}%)")

## Part 12: Latency Summary

In [ ]:
latencies = [r["latency"] for r in results]

print("Latency Summary")
print("=" * 40)
print(f"  Min:    {min(latencies):.2f}s")
print(f"  Max:    {max(latencies):.2f}s")
print(f"  Mean:   {np.mean(latencies):.2f}s")
print(f"  Median: {np.median(latencies):.2f}s")
print(f"  P95:    {np.percentile(latencies, 95):.2f}s")

target = 5.0
within_target = sum(1 for l in latencies if l < target)
print(f"\n  Within {target}s target: {within_target}/{len(latencies)} ({within_target/len(latencies)*100:.0f}%)")

---

## Final Scorecard

Run this cell to see your overall score.

In [ ]:
print("\n" + "=" * 60)
print("CAPSTONE SCORECARD")
print("=" * 60)

# Collect metrics
metrics = {
    "Faithfulness (RAGAS)": ragas_result.get("faithfulness", 0),
    "Answer Relevancy (RAGAS)": ragas_result.get("answer_relevancy", 0),
    "Context Precision (RAGAS)": ragas_result.get("context_precision", 0),
    "Context Recall (RAGAS)": ragas_result.get("context_recall", 0),
    "Source Accuracy": correct_source / total,
    "Latency (% under 5s)": within_target / len(latencies),
}

overall = np.mean(list(metrics.values()))

for name, score in metrics.items():
    bar = "#" * int(score * 20)
    status = "PASS" if score >= 0.7 else "NEEDS WORK"
    print(f"  {name:<30} {score:.2f}  {bar:<20} {status}")

print(f"\n  {'OVERALL':<30} {overall:.2f}")
print(f"\n  Grade: {'A' if overall >= 0.9 else 'B' if overall >= 0.8 else 'C' if overall >= 0.7 else 'D'}")

if overall >= 0.8:
    print("\n  Congratulations! Your production RAG pipeline meets quality standards.")
else:
    print("\n  Your pipeline needs improvement. Focus on the metrics marked 'NEEDS WORK'.")

## Improvement Ideas (If You Have Time)

1. **Semantic caching** — Use embedding similarity to match "near-identical" queries to cache entries
2. **Chunk metadata enrichment** — Add section headers as metadata for better filtering
3. **Query classification** — Route queries to specific sources before retrieval
4. **Answer compression** — Summarize long answers for conciseness
5. **Streaming** — Stream the answer token by token for better UX
6. **Guardrails** — Add input/output validation to prevent off-topic or harmful responses

## Key Takeaways

1. **Production RAG is more than retrieval + generation.** Hybrid search, re-ranking, caching, and evaluation are all essential.
2. **Evaluation is non-negotiable.** Without metrics like RAGAS, you're flying blind.
3. **A golden test set** is your safety net — run it after every change to catch regressions.
4. **Caching** dramatically reduces latency and cost for repeated queries.
5. **Source attribution** builds trust — users need to verify answers against originals.
6. **There is no perfect RAG pipeline.** Every choice (chunk size, retrieval strategy, model) is a trade-off. Measure, iterate, improve.